# L06 · 并行仿真与批量 Franka 控制

本实验在 L05 目标到达链前增加前导环境维度：

```text
一份 Plane + Franka 拓扑 → B=4 份状态 → 批量 IK/control
→ 选择性更新环境 1 和 3 → 逐环境证据
```

关闭渲染时仍能完成完整数值路径。可选相机分支会检查真实的批量 RGB/depth array，但不能代替 IK 或控制检查。


## 运行前准备

`ROBO_GENESIS_BACKEND=auto` 会在可用时选择已验证的 AMD backend，否则使用 CPU。最低路径可设为 `cpu`。`ROBO_GENESIS_RENDER=0` 不创建相机，但会运行完整数值实验；启动 kernel 前设为 `1`，则要求获得真实的 environment-separated RGB/depth batch。改变任一设置后都要重启 kernel。

先预测：

1. `scene.build(n_envs=4)` 之后的 qpos shape 是什么？
2. `env_spacing` 会改变世界坐标系 IK target 吗？
3. 使用 `envs_idx=[1, 3]` 时，command row 0 属于哪个环境？
4. 在这次选择性更新中，环境 0 和 2 会停止演化吗？
5. 4 幅 camera image 能证明 4 个 IK candidate 都通过了吗？


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.scene_config import (
    FRANKA_FORCE_MAX,
    FRANKA_FORCE_MIN,
    FRANKA_KP,
    FRANKA_KV,
    FRANKA_MJCF,
    FRANKA_QPOS,
)

lesson = load_course_manifest().lesson("L06")
assert lesson.slug == "parallel-simulation-and-batched-franka-control"
assert lesson.status.value == "planned"

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l06-parallel-batched-franka", show_viewer=False)
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == "cpu" else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")

if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)

print("Genesis:", environment["genesis_world"])
print("requested backend:", backend_mode)
print("actual backend:", actual_backend)
print("render enabled:", render_enabled)
print("output directory:", runtime["output_dir"].resolve())


## 从一份拓扑构建 4 份状态

Plane、Franka 和可选 camera 都只声明一次。`scene.build(n_envs=4)` 会创建 4 份带前导环境维度的独立状态副本。`env_spacing` 和 `n_envs_per_row` 只影响可视化；下文任何 IK target 都没有包含布局 offset。

只有请求渲染时才会在 build 前声明 camera。`env_separate_rigid=True` 要求 Rasterizer 为 `rendered_envs_idx` 中的每个环境分别返回图像。


In [ ]:
B = 4
DT = 0.01
SUBSTEPS = 2
CONTROL_STEPS = 180
IK_POSITION_TOLERANCE = 5e-4
IK_ROTATION_TOLERANCE = 5e-3
DYNAMIC_POSITION_TOLERANCE = 0.02
DYNAMIC_ORIENTATION_TOLERANCE = 0.05
UNTOUCHED_MOTION_TOLERANCE = 0.005
RETAINED_ERROR_INCREASE_TOLERANCE = 0.002
CAMERA_RESOLUTION = (640, 360)
RENDERED_ENVS = [0, 1, 2, 3]

scene_options = {
    "sim_options": gs.options.SimOptions(dt=DT, substeps=SUBSTEPS),
    "show_viewer": False,
}
if render_enabled:
    scene_options["vis_options"] = gs.options.VisOptions(
        rendered_envs_idx=RENDERED_ENVS,
        env_separate_rigid=True,
        shadow=False,
        plane_reflection=False,
    )

scene = gs.Scene(**scene_options)
scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(gs.morphs.MJCF(file=FRANKA_MJCF))
camera = None
if render_enabled:
    camera = scene.add_camera(
        res=CAMERA_RESOLUTION,
        pos=(1.6, 1.6, 1.2),
        lookat=(0.25, 0.0, 0.35),
        fov=48,
        GUI=False,
    )
scene.build(n_envs=B, env_spacing=(1.1, 1.1), n_envs_per_row=2)

hand = franka.get_link("hand")
joint_names = [f"joint{i}" for i in range(1, 8)] + [
    "finger_joint1",
    "finger_joint2",
]
all_dofs = np.asarray(
    [franka.get_joint(name).dofs_idx_local[0] for name in joint_names],
    dtype=int,
)
arm_dofs = all_dofs[:7]
q_start = np.tile(np.asarray(FRANKA_QPOS, dtype=float), (B, 1))

franka.set_dofs_kp(np.asarray(FRANKA_KP), dofs_idx_local=all_dofs)
franka.set_dofs_kv(np.asarray(FRANKA_KV), dofs_idx_local=all_dofs)
franka.set_dofs_force_range(
    np.asarray(FRANKA_FORCE_MIN),
    np.asarray(FRANKA_FORCE_MAX),
    dofs_idx_local=all_dofs,
)
franka.set_dofs_position(
    q_start,
    dofs_idx_local=all_dofs,
    zero_velocity=True,
)

initial_qpos = to_numpy(franka.get_qpos()).astype(float)
assert initial_qpos.shape == (B, 9)
assert arm_dofs.shape == (7,)
print("qpos shape after batched build:", initial_qpos.shape)
print("arm DOF indices:", arm_dofs.tolist())
print("visualization spacing only: (1.1, 1.1)")
print("camera:", "batched camera declared" if render_enabled else "SKIP")


## 显式写出 shape 与验收合同

完整批量的 q、position、quaternion 和 IK residual 分别为 `(4, 9)`、`(4, 3)`、`(4, 4)` 与 `(4, 6)`。后面的选择性阶段使用相同的量，但前导维度变为 2。每个 quaternion row 都采用 Genesis 的 `wxyz` 顺序，每个 residual row 都要单独检查。

下面的 helper 只负责逐行归一化 quaternion、读取全部 hand pose、计算最短角误差和展开逐行 IK 验收。后续 cell 仍会直接展示 target 构造、index、IK、control 与 stepping。


In [ ]:
def normalize_wxyz_rows(quaternions, expected_rows):
    quaternions = np.asarray(quaternions, dtype=float)
    if quaternions.shape != (expected_rows, 4):
        raise ValueError(
            f"expected quaternion shape {(expected_rows, 4)}, got {quaternions.shape}"
        )
    if not np.isfinite(quaternions).all():
        raise ValueError("quaternions contain non-finite values")
    norms = np.linalg.norm(quaternions, axis=1, keepdims=True)
    if np.any(norms < 1e-12):
        raise ValueError("a zero quaternion does not define an orientation")
    return quaternions / norms


def quaternion_angle_errors(measured_wxyz, target_wxyz):
    measured_wxyz = np.asarray(measured_wxyz, dtype=float)
    target_wxyz = np.asarray(target_wxyz, dtype=float)
    if measured_wxyz.shape != target_wxyz.shape or measured_wxyz.ndim != 2:
        raise ValueError("quaternion batches must have matching shapes")
    measured = normalize_wxyz_rows(measured_wxyz, measured_wxyz.shape[0])
    target = normalize_wxyz_rows(target_wxyz, target_wxyz.shape[0])
    cosine_half_angle = np.clip(
        np.abs(np.sum(measured * target, axis=1)),
        0.0,
        1.0,
    )
    return 2.0 * np.arccos(cosine_half_angle)


def read_hand_poses():
    positions = to_numpy(hand.get_pos(relative=False)).astype(float)
    quaternions = to_numpy(hand.get_quat(relative=False)).astype(float)
    if positions.shape != (B, 3) or quaternions.shape != (B, 4):
        raise AssertionError(
            f"unexpected hand pose shapes: {positions.shape}, {quaternions.shape}"
        )
    if not np.isfinite(positions).all() or not np.isfinite(quaternions).all():
        raise AssertionError("hand poses contain non-finite values")
    return positions, quaternions


def summarize_ik(q_raw, error_raw, envs_idx, label):
    envs_idx = np.asarray(envs_idx, dtype=int)
    q = to_numpy(q_raw).astype(float)
    error = to_numpy(error_raw).astype(float)
    expected_rows = len(envs_idx)
    if q.shape != (expected_rows, 9) or error.shape != (expected_rows, 6):
        raise AssertionError(
            f"{label}: unexpected q/error shapes {q.shape}, {error.shape}"
        )
    position_residuals = np.linalg.norm(error[:, :3], axis=1)
    rotation_residuals = np.linalg.norm(error[:, 3:], axis=1)
    valid = (
        np.isfinite(q).all(axis=1)
        & np.isfinite(error).all(axis=1)
        & (position_residuals <= IK_POSITION_TOLERANCE)
        & (rotation_residuals <= IK_ROTATION_TOLERANCE)
    )
    for row, env_index in enumerate(envs_idx):
        print(
            f"{label} row {row} -> env {env_index}: "
            f"position residual={position_residuals[row]:.6f} m; "
            f"rotation residual={rotation_residuals[row]:.6f} rad; "
            f"accepted={bool(valid[row])}"
        )
    return q, error, position_residuals, rotation_residuals, valid


TARGET_POSITIONS = np.array([
    [0.42, -0.12, 0.35],
    [0.48, -0.04, 0.40],
    [0.48,  0.06, 0.32],
    [0.40,  0.14, 0.38],
])
TARGET_QUATERNIONS = normalize_wxyz_rows(
    np.tile(np.array([0.0, 1.0, 0.0, 0.0]), (B, 1)),
    B,
)
initial_positions, initial_quaternions = read_hand_poses()

assert TARGET_POSITIONS.shape == (B, 3)
assert TARGET_QUATERNIONS.shape == (B, 4)
assert np.isfinite(TARGET_POSITIONS).all()
assert np.allclose(np.linalg.norm(TARGET_QUATERNIONS, axis=1), 1.0)
print("full-batch shape ledger:", {
    "q": initial_qpos.shape,
    "position": TARGET_POSITIONS.shape,
    "quaternion": TARGET_QUATERNIONS.shape,
    "expected residual": (B, 6),
})
print("selective shape ledger:", {
    "q": (2, 9),
    "position": (2, 3),
    "quaternion": (2, 4),
    "residual": (2, 6),
})


## 求解并验收全部 4 行 IK

下一次调用为每个环境各分配一行世界坐标系 target。有限的 q 只是 candidate。只有 4 行都满足准确的 `(4, 9)` / `(4, 6)` shape 合同和两项 residual threshold，才会发送 command。Mean 或 maximum 可以汇总各行，但不能代替逐行检查。


In [ ]:
ALL_ENVS = np.arange(B, dtype=int)
q_goal_raw, ik_error_raw = franka.inverse_kinematics(
    link=hand,
    pos=TARGET_POSITIONS,
    quat=TARGET_QUATERNIONS,
    init_qpos=q_start,
    dofs_idx_local=arm_dofs,
    respect_joint_limit=True,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    return_error=True,
)
(
    q_goal,
    ik_error,
    ik_position_residuals,
    ik_rotation_residuals,
    full_ik_valid,
) = summarize_ik(q_goal_raw, ik_error_raw, ALL_ENVS, "baseline IK")

if not np.all(full_ik_valid):
    raise AssertionError("at least one baseline IK row failed acceptance")
print("baseline IK accepted for every environment")


## 执行完整批量并测量动力学结果

通过验收的 IK row 会成为 PD position target，不会瞬移状态。循环每次发送 `(4, 9)` target，调用一次 `scene.step()`，再读取全部 4 个世界坐标系 hand pose。最终 position 与 orientation error 按环境检查，并且每个 position error 都必须比各自的初值更小。


In [ ]:
baseline_position_history = [initial_positions.copy()]
baseline_quaternion_history = [initial_quaternions.copy()]
for _ in range(CONTROL_STEPS):
    franka.control_dofs_position(q_goal, dofs_idx_local=all_dofs)
    scene.step()
    positions, quaternions = read_hand_poses()
    baseline_position_history.append(positions.copy())
    baseline_quaternion_history.append(quaternions.copy())

baseline_position_history = np.asarray(baseline_position_history)
baseline_quaternion_history = np.asarray(baseline_quaternion_history)
baseline_position_error_history = np.linalg.norm(
    baseline_position_history - TARGET_POSITIONS[None, :, :],
    axis=2,
)
baseline_orientation_error_history = np.asarray([
    quaternion_angle_errors(quaternions, TARGET_QUATERNIONS)
    for quaternions in baseline_quaternion_history
])
baseline_positions = baseline_position_history[-1].copy()
baseline_quaternions = baseline_quaternion_history[-1].copy()
baseline_position_errors = baseline_position_error_history[-1].copy()
baseline_orientation_errors = baseline_orientation_error_history[-1].copy()

baseline_dynamic_valid = (
    np.isfinite(baseline_position_history).all(axis=(0, 2))
    & np.isfinite(baseline_quaternion_history).all(axis=(0, 2))
    & (baseline_position_errors < DYNAMIC_POSITION_TOLERANCE)
    & (baseline_orientation_errors < DYNAMIC_ORIENTATION_TOLERANCE)
    & (baseline_position_errors < baseline_position_error_history[0])
)
for env_index in ALL_ENVS:
    print(
        f"env {env_index}: final position error="
        f"{baseline_position_errors[env_index]:.6f} m; "
        f"orientation error={baseline_orientation_errors[env_index]:.6f} rad; "
        f"accepted={bool(baseline_dynamic_valid[env_index])}"
    )
if not np.all(baseline_dynamic_valid):
    raise AssertionError("at least one baseline dynamic row failed acceptance")


## 只更新环境 1 和 3

`SELECTED_ENVS=[1, 3]` 只需要两行 target、IK 与 command：局部 row 0 映射到环境 1，局部 row 1 映射到环境 3。环境 0 和 2 没有收到新 target，但它们并未冻结。每次 `scene.step()` 推进整个 Scene 时，它们会继续在保留的 baseline PD target 下演化。

因此，检查时应让选中环境与新 target 比较，让未选环境与旧 target 比较，并同时报告 untouched motion 和 retained-target error change。最后一张图把 baseline 与 selective position-error history 并列展示；这是数值轨迹证据，不是 Genesis render。


In [ ]:
SELECTED_ENVS = np.array([1, 3], dtype=int)
UNTOUCHED_ENVS = np.array([0, 2], dtype=int)
SELECTIVE_TARGET_POSITIONS = np.array([
    [0.44, -0.16, 0.32],  # row 0 -> environment 1
    [0.50,  0.12, 0.40],  # row 1 -> environment 3
])
SELECTIVE_TARGET_QUATERNIONS = normalize_wxyz_rows(
    TARGET_QUATERNIONS[SELECTED_ENVS],
    len(SELECTED_ENVS),
)

assert SELECTIVE_TARGET_POSITIONS.shape == (len(SELECTED_ENVS), 3)
assert SELECTIVE_TARGET_QUATERNIONS.shape == (len(SELECTED_ENVS), 4)
assert np.isfinite(SELECTIVE_TARGET_POSITIONS).all()
assert np.allclose(np.linalg.norm(SELECTIVE_TARGET_QUATERNIONS, axis=1), 1.0)
for row, env_index in enumerate(SELECTED_ENVS):
    print(
        f"selective target row {row} -> env {env_index}: "
        f"{SELECTIVE_TARGET_POSITIONS[row]}"
    )

q_selected_raw, selected_ik_error_raw = franka.inverse_kinematics(
    link=hand,
    pos=SELECTIVE_TARGET_POSITIONS,
    quat=SELECTIVE_TARGET_QUATERNIONS,
    dofs_idx_local=arm_dofs,
    respect_joint_limit=True,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    return_error=True,
    envs_idx=SELECTED_ENVS,
)
(
    q_selected,
    selected_ik_error,
    selected_ik_position_residuals,
    selected_ik_rotation_residuals,
    selected_ik_valid,
) = summarize_ik(
    q_selected_raw,
    selected_ik_error_raw,
    SELECTED_ENVS,
    "selective IK",
)
if not np.all(selected_ik_valid):
    raise AssertionError("at least one selective IK row failed acceptance")

positions_before_selective = baseline_positions.copy()
selective_position_history = [positions_before_selective.copy()]
selective_quaternion_history = [baseline_quaternions.copy()]
for _ in range(CONTROL_STEPS):
    franka.control_dofs_position(
        q_selected,
        dofs_idx_local=all_dofs,
        envs_idx=SELECTED_ENVS,
    )
    scene.step()
    positions, quaternions = read_hand_poses()
    selective_position_history.append(positions.copy())
    selective_quaternion_history.append(quaternions.copy())

selective_position_history = np.asarray(selective_position_history)
selective_quaternion_history = np.asarray(selective_quaternion_history)
stage_two_target_positions = TARGET_POSITIONS.copy()
stage_two_target_quaternions = TARGET_QUATERNIONS.copy()
stage_two_target_positions[SELECTED_ENVS] = SELECTIVE_TARGET_POSITIONS
stage_two_target_quaternions[SELECTED_ENVS] = SELECTIVE_TARGET_QUATERNIONS
selective_position_error_history = np.linalg.norm(
    selective_position_history - stage_two_target_positions[None, :, :],
    axis=2,
)
selective_orientation_error_history = np.asarray([
    quaternion_angle_errors(quaternions, stage_two_target_quaternions)
    for quaternions in selective_quaternion_history
])
final_positions = selective_position_history[-1]
final_quaternions = selective_quaternion_history[-1]
selected_position_errors = selective_position_error_history[-1, SELECTED_ENVS]
selected_orientation_errors = selective_orientation_error_history[-1, SELECTED_ENVS]
retained_position_errors = selective_position_error_history[-1, UNTOUCHED_ENVS]
retained_orientation_errors = selective_orientation_error_history[-1, UNTOUCHED_ENVS]
untouched_motion = np.linalg.norm(
    final_positions[UNTOUCHED_ENVS]
    - positions_before_selective[UNTOUCHED_ENVS],
    axis=1,
)
retained_error_change = (
    retained_position_errors - baseline_position_errors[UNTOUCHED_ENVS]
)

stage_two_finite = (
    np.isfinite(selective_position_history).all(axis=(0, 2))
    & np.isfinite(selective_quaternion_history).all(axis=(0, 2))
)
selected_dynamic_valid = (
    stage_two_finite[SELECTED_ENVS]
    & (selected_position_errors < DYNAMIC_POSITION_TOLERANCE)
    & (selected_orientation_errors < DYNAMIC_ORIENTATION_TOLERANCE)
)
retained_dynamic_valid = (
    stage_two_finite[UNTOUCHED_ENVS]
    & (retained_position_errors < DYNAMIC_POSITION_TOLERANCE)
    & (retained_orientation_errors < DYNAMIC_ORIENTATION_TOLERANCE)
    & (untouched_motion < UNTOUCHED_MOTION_TOLERANCE)
    & (retained_error_change <= RETAINED_ERROR_INCREASE_TOLERANCE)
)
for row, env_index in enumerate(SELECTED_ENVS):
    print(
        f"selected env {env_index}: position error="
        f"{selected_position_errors[row]:.6f} m; orientation error="
        f"{selected_orientation_errors[row]:.6f} rad; "
        f"accepted={bool(selected_dynamic_valid[row])}"
    )
for row, env_index in enumerate(UNTOUCHED_ENVS):
    print(
        f"untouched env {env_index}: retained position error="
        f"{retained_position_errors[row]:.6f} m; retained orientation error="
        f"{retained_orientation_errors[row]:.6f} rad; motion="
        f"{untouched_motion[row]:.6f} m; error change="
        f"{retained_error_change[row]:+.6f} m; "
        f"accepted={bool(retained_dynamic_valid[row])}"
    )
if not np.all(selected_dynamic_valid) or not np.all(retained_dynamic_valid):
    raise AssertionError("selective update or retained-target checks failed")

time = np.arange(CONTROL_STEPS + 1) * DT
figure, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for env_index in ALL_ENVS:
    axes[0].plot(time, baseline_position_error_history[:, env_index], label=f"env {env_index}")
    target_kind = "new" if env_index in SELECTED_ENVS else "retained"
    axes[1].plot(
        time,
        selective_position_error_history[:, env_index],
        label=f"env {env_index} ({target_kind})",
    )
for axis, title in zip(axes, ["Baseline targets", "Selective update"]):
    axis.axhline(DYNAMIC_POSITION_TOLERANCE, color="black", linestyle="--")
    axis.set(xlabel="time [s]", ylabel="position error [m]", title=title)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
figure.tight_layout()
plt.show()


## 可选批量相机与最终检查

启用渲染后，由于 `rendered_envs_idx` 包含 4 个环境且设置了 `env_separate_rigid=True`，一个 Rasterizer camera 会返回 RGB `(4, H, W, 3)` 和 depth `(4, H, W)`。2×2 mosaic 只是这些真实 array 的一种查看方式。关闭渲染时，该分支会明确报告 `SKIP`，不会创建替代图像。

最终检查继续区分四类主张：batching correctness、simulation throughput、batched rendering 与完整 parallel recording。本 notebook 验证第一类，并在启用时验证第三类；它不报告固定 speedup，也不实现 recorder。


In [ ]:
camera_path_ok = True
camera_status = "SKIP — ROBO_GENESIS_RENDER=0; no camera was created"
if render_enabled:
    rgb, depth, _, _ = camera.render(rgb=True, depth=True)
    rgb = to_numpy(rgb)
    depth = to_numpy(depth)
    width, height = CAMERA_RESOLUTION
    rgb_shape_ok = rgb.shape == (len(RENDERED_ENVS), height, width, 3)
    depth_shape_ok = depth.shape == (len(RENDERED_ENVS), height, width)
    camera_checks = {
        "rgb_shape": rgb_shape_ok,
        "depth_shape": depth_shape_ok,
        "rgb_dtype": rgb.dtype == np.uint8,
        "depth_float": np.issubdtype(depth.dtype, np.floating),
        "pixels_finite": np.isfinite(rgb).all() and np.isfinite(depth).all(),
        "rgb_variation": rgb_shape_ok and bool(np.all(np.std(rgb, axis=(1, 2, 3)) > 0)),
        "positive_depth": depth_shape_ok and all(
            np.count_nonzero(image > 0) > 0 for image in depth
        ),
    }
    camera_path_ok = all(camera_checks.values())
    if not camera_path_ok:
        raise AssertionError(camera_checks)
    camera_status = "PASSED — batched RGB/depth captured"
    print("RGB:", rgb.shape, rgb.dtype)
    print("depth:", depth.shape, depth.dtype)

    figure, axes = plt.subplots(2, 2, figsize=(9, 6))
    for row, env_index in enumerate(RENDERED_ENVS):
        axes.flat[row].imshow(rgb[row])
        axes.flat[row].set_title(f"Rendered environment {env_index}")
        axes.flat[row].axis("off")
    figure.tight_layout()
    plt.show()

print(camera_status)

final_checks = {
    "genesis_1_3_3": environment["genesis_world"] == "1.3.3",
    "actual_backend_supported": actual_backend in {"cpu", "amdgpu"},
    "forced_cpu_honored": backend_mode != "cpu" or actual_backend == "cpu",
    "batched_q_shape": initial_qpos.shape == (B, 9),
    "full_target_shapes": (
        TARGET_POSITIONS.shape == (B, 3)
        and TARGET_QUATERNIONS.shape == (B, 4)
    ),
    "full_ik_rows": bool(np.all(full_ik_valid)),
    "baseline_dynamic_rows": bool(np.all(baseline_dynamic_valid)),
    "selective_mapping": (
        np.array_equal(SELECTED_ENVS, np.array([1, 3]))
        and q_selected.shape == (2, 9)
        and selected_ik_error.shape == (2, 6)
    ),
    "selective_ik_rows": bool(np.all(selected_ik_valid)),
    "selected_dynamic_rows": bool(np.all(selected_dynamic_valid)),
    "retained_target_rows": bool(np.all(retained_dynamic_valid)),
    "camera_branch": camera_path_ok,
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError("L06 checks failed: " + ", ".join(failed))

print("camera:", camera_status)
print("L06 CHECK: PASSED")


## 检查点与练习

根据本次运行逐行打印的值回答：

1. 为什么 `n_envs=0` 没有前导环境维度，而 `n_envs=1` 有？
2. 为什么没有为任何 target 添加 visualization offset？
3. 哪些证据分别验收 baseline IK candidate 与动态结果？
4. 为什么 selective row 0 映射到环境 1，而不是环境 0？
5. 为什么环境 0 和 2 必须同时检查 motion 与 retained-target error change？
6. 哪些关于 throughput、recording、path safety 或 grasp success 的主张仍未验证？

作为小练习，只把 `SELECTED_ENVS` 改成 `[0, 2]`、`UNTOUCHED_ENVS` 改成 `[1, 3]`，并把两行 selected position 改为 `[0.46, -0.08, 0.38]` 和 `[0.44, 0.10, 0.36]`。先预测每个 subset shape 与行映射，再重新执行 selected/untouched 检查。保持 B、orientation、controller setting、step count 和 rendering mode 不变。

L07 会在保留这套控制与索引纪律的同时加入任务几何；L09 会把 environment identity 和前导维度用于数据记录，但完整 parallel recorder 仍不属于本实验。
